In [69]:
class Value:
    def __init__(self,data,parents=set(),operation="",label=""):
        self.data=data
        self._prev=parents
        self._op=operation
        self.label=label
        self._backward=lambda :None
        self.grad=0
    def __repr__(self):
        return f"Value({self.data}, {self.label})"
    def __add__(self,other):
        out=Value(self.data+other.data,(self,other),operation="+")
        def backward():
            self.grad+=out.grad
            other.grad+=out.grad
        out._backward=backward
        return out
    
    def __radd__(self, other):
        return self + other
    def __mul__(self,other):
        out=Value(self.data*other.data,(self.label,other),operation="*")
        def backward():
            self.grad+=out.grad*other.data
            other.grad+=out.grad*self.data
        out._backward=backward
        return out
    def __rmul__(self, other):
        return self * other

    def __neg__(self):
        out=Value(-1*self.data,(self),operation="Negation")
        def backward():
            self.grad+=-1*out.grad
        return out
    def __sub__(self,other):
        out=Value(self.data-other.data,(self.label,other.label),operation="+")
        def backward():
            self.grad+=out.grad
            other.grad+=out.grad
        out._backward=backward
        return out


In [53]:
a=Value(2);a.label="a"
b=Value(3);b.label="b"
e=Value(4);e.label="e"

In [68]:
c=a*b;c.label="c"
d=c+e;d.label="d"
f=c*a;f.label="f"
L=f+f;L.label="L"
L.grad=1

In [64]:
L._backward()
f._backward()
c._backward()

In [65]:
a.grad,b.grad,e.grad,c.grad,f.grad,L.grad

(24, 8, 0, 4, 2, 1)

In [ ]:
def backward(Loss):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(Loss)
        Loss.grad = 1